# RAG com IA e PDFs — versão de notebook, adaptada de um app real

Curso: IA Aplicada ao Desenvolvimento de Software para Servidores Públicos · Aula 8 (RAG)

Este notebook adapta, para o Colab, um app Streamlit real ("Trabalho de RAG com IA") que lê um PDF, monta um prompt com o texto inteiro do documento e a pergunta, e usa a API da OpenAI para responder — com histórico de perguntas mantido na conversa. Aqui, adaptamos para os **dois** Relatórios de Gestão da Enap (2023 e 2024) que a aula já usa.

**Diferença importante em relação a um RAG "completo":** este notebook não faz chunking, embeddings nem retrieval — ele extrai o texto inteiro do PDF e cola no prompt, junto com a pergunta. Funciona bem para documentos que cabem na janela de contexto do modelo (é o caso aqui — veja os números reais no Passo 2), mas não escala para um acervo de centenas de documentos, e reenvia o texto inteiro a cada pergunta, o que custa mais tokens do que buscar só o trecho relevante (a técnica das Etapas 1 e 2 desta aula, e do notebook `rag_enap_colab_embeddings.ipynb`, se você quiser comparar as duas abordagens lado a lado).

## Passo 0 — instalar as bibliotecas

As mesmas três do app original:
- `PyPDF2` — lê o texto de dentro de um PDF.
- `openai` — chama a API da OpenAI para gerar a resposta.
- `python-dotenv` — carrega a chave de API de um arquivo `.env`, em vez de deixá-la escrita no código.

In [ ]:
!pip install -q PyPDF2 openai python-dotenv

## Passo 1 — enviar os dois relatórios

**Antes de rodar a célula abaixo**, baixe os dois PDFs no seu computador — pelos botões "Baixar Relatório de Gestão 2023/2024" na própria aula, ou direto destes links: [Relatório 2023](https://repositorio.enap.gov.br/handle/1/9959) e [Relatório 2024](https://repositorio.enap.gov.br/handle/1/8854).

**Por que upload, e não a célula buscar sozinha**: o servidor do repositório da Enap bloqueia pedidos automáticos vindos de faixas de IP de datacenter do Google (`HTTPError: 403 Forbidden`, mesmo o link funcionando normalmente em qualquer navegador). Pedir upload evita esse problema por completo.

Rode a célula abaixo e selecione os dois PDFs baixados de uma vez (Ctrl/Cmd + clique para marcar os dois).

In [ ]:
from google.colab import files

print("Envie os dois arquivos baixados: o Relatório de Gestão 2023 e o Relatório de Gestão 2024 da Enap.")
enviados = files.upload()

pdf_bytes_por_ano = {}
for nome_arquivo, conteudo in enviados.items():
    if "2023" in nome_arquivo:
        pdf_bytes_por_ano["2023"] = conteudo
    elif "2024" in nome_arquivo:
        pdf_bytes_por_ano["2024"] = conteudo
    else:
        print(f"Aviso: não reconheci '{nome_arquivo}' como 2023 ou 2024 pelo nome do arquivo.")

faltando = {"2023", "2024"} - set(pdf_bytes_por_ano.keys())
if faltando:
    raise RuntimeError(
        f"Envie os relatórios de {' e '.join(sorted(faltando))} para continuar "
        "(mantenha '2023'/'2024' em algum lugar do nome do arquivo)."
    )

print("Recebidos:", list(pdf_bytes_por_ano.keys()))

## Passo 2 — extrair o texto (a mesma função `extrair_texto_pdf` do app original, adaptada)

No app Streamlit, essa função lia um arquivo do disco (`pasta/arquivo.pdf`); aqui, lê os bytes já recebidos no upload. O resto é idêntico: `PyPDF2.PdfReader`, um laço por página, concatenando o texto.

In [ ]:
import io
import PyPDF2

def extrair_texto_pdf(conteudo_pdf):
    texto = ""
    leitor = PyPDF2.PdfReader(io.BytesIO(conteudo_pdf))
    for pagina in range(len(leitor.pages)):
        texto_pagina = leitor.pages[pagina].extract_text() or ""  # páginas sem texto extraível não quebram a extração
        texto += texto_pagina + "
"
    return texto

texto_por_ano = {ano: extrair_texto_pdf(conteudo) for ano, conteudo in pdf_bytes_por_ano.items()}

for ano, texto in texto_por_ano.items():
    print(f"Relatório {ano}: {len(texto)} caracteres extraídos (~{len(texto)//4} tokens)")

**Números reais, medidos antes de publicar este notebook:** o relatório de 2023 tem ~99.600 caracteres (~25.000 tokens); o de 2024, ~174.300 caracteres (~43.600 tokens). Juntos, ~68.500 tokens — cabe tranquilamente na janela de contexto do `gpt-4o-mini` (128.000 tokens), mas é um bloco grande para reenviar a cada pergunta. É esse custo, por pergunta, que a técnica de chunking + embeddings + retrieval (Etapas 1 e 2 da aula) evita — ela manda só os 3-4 trechos relevantes, não o documento inteiro.

## Passo 3 — configurar a chave de API com `python-dotenv`

No app original, a chave vem de um arquivo `.env` na mesma pasta do projeto (`OPENAI_API_KEY=sk-...`), carregado com `load_dotenv()` — o mesmo padrão usado em qualquer projeto Python real, inclusive fora do Colab. Aqui, criamos esse `.env` na hora, a partir de uma chave colada com `getpass` (não fica visível na tela nem salva no notebook em texto puro).

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

chave_api = getpass("Cole sua chave da API OpenAI: ")

with open(".env", "w") as f:
    f.write(f"OPENAI_API_KEY={chave_api}\n")

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise RuntimeError("A chave da API OpenAI não foi encontrada. Rode a célula de novo.")

print("Chave carregada do .env — pronta para uso.")

## Passo 4 — perguntar (a mesma lógica do botão "Pesquisar" do app original)

O app original: pega o texto do PDF selecionado + o histórico da conversa + a nova pergunta, monta um prompt único, e chama `client.chat.completions.create(model="gpt-4o-mini", ...)`. Aqui fazemos igual, mas com a opção de incluir um relatório, o outro, ou os dois juntos — útil para perguntas que cruzam os dois anos (o mesmo tipo de pergunta do Desafio final desta aula).

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
historico = []

def montar_documento(anos):
    partes = [f"[Relatório de Gestão Enap {ano}]\n{texto_por_ano[ano]}" for ano in anos]
    return "\n\n".join(partes)

def perguntar(pergunta, anos=("2023", "2024")):
    documento = montar_documento(anos)
    contexto = "\n".join(
        f"Pergunta: {h['pergunta']}\nResposta: {h['resposta']}" for h in historico
    )
    prompt = (
        f"Baseado no(s) seguinte(s) documento(s):\n\n{documento}\n\n"
        f"Histórico da conversa:\n{contexto}\n\n"
        f"Pergunta: {pergunta}\nResposta:"
    )
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
    )
    resposta = completion.choices[0].message.content
    historico.append({"pergunta": pergunta, "resposta": resposta})
    return resposta

resposta = perguntar("Quantos participantes o Congresso do CLAD reuniu, e em qual ano isso aconteceu?")
print(resposta)

## Passo 5 — uma segunda pergunta, testando o histórico

Como no app original, o histórico entra no prompt a cada nova pergunta — o modelo "lembra" da troca anterior dentro da mesma sessão. Teste com uma pergunta que só faz sentido sabendo da resposta de cima (ex.: "e no outro relatório, algo parecido aconteceu?").

In [ ]:
resposta2 = perguntar("E no outro relatório, teve algum evento de porte parecido? Cite o nome e o número de participantes.")
print(resposta2)

## O que este notebook mostra (e o que ele não faz)

1. **Funciona de ponta a ponta** — dois PDFs reais, upload, extração com `PyPDF2`, prompt com histórico, resposta gerada pela OpenAI. É o mesmo app do trabalho, só que em células de notebook em vez de em uma página Streamlit.
2. **Não é RAG com retrieval** — não há chunking, embeddings nem busca por similaridade; o documento inteiro (ou os dois) vai no prompt sempre. Funciona aqui porque os relatórios cabem na janela de contexto do modelo — mas o custo por pergunta é maior, e a abordagem não escalaria para um acervo com muito mais documentos.
3. **Cross-document funciona "de graça"** — como o texto dos dois relatórios pode ir junto no mesmo prompt, perguntas que cruzam os dois anos não precisam de nenhum mecanismo extra (diferente do retrieval, que precisaria recuperar trechos dos dois documentos separadamente).
4. **Publicar como app de verdade**: o arquivo `app.py` nesta mesma pasta (`aula-8/codigo/rag-exemplo/app.py`) é a versão Streamlit completa desta lógica, pronta para rodar com `streamlit run app.py` ou publicar no Streamlit Community Cloud — veja `PUBLICAR.md` para o passo a passo de configurar `.env`, subir para o GitHub e publicar.